# Gaussian Approximation:

This notebook keeps HOLYKALE's self-consistency iteration visible. It defaults to no execution because one trial is 500 seconds at 0.1 ms resolution.

## Self-consistent iterative scheme

The recurrent input has mean

$$\mu = R_I + C_E J\tau(1-\gamma g)\nu,$$

with colored-noise spectrum generated from the preceding single-neuron spectrum using

$$S_\eta(f)=\tau^2[C_EJ^2+C_I(gJ)^2]S_x(f).$$

The first generation uses independent Poisson E/I input. Later generations use HOLYKALE's random-phase colored-noise construction. No damping or early stopping is enabled.

## Imports, parameters, and random seeds

In [ ]:
from pathlib import Path
import json
import platform
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
from scipy import __version__ as scipy_version
from scipy.signal import welch

RUN_SIMULATION = False
initial_rate_hz = 15.0             # select 15.0 or 71.0 Hz
n_gen = 15                          # generations
SEED = 4321
RUN_ID = f"holy kale-gaussian-{int(initial_rate_hz)}hz"
SAVE_FIGURES = True
T_trial = 500.0       # seconds
dt_sec = 0.0001       # seconds (0.1 ms)
N_steps = int(T_trial / dt_sec)
SMALL_TEST_CONFIG = {"T_trial_s": 1.0, "dt_s": 0.0001, "generations": 2}
## SMALL_TEST_CONFIG is for explicit smoke/equivalence checks only; it does not replace the defaults.

# Original dimensionless SI-valued parameters.
tau_val = 0.020
vT_val = 0.020
vR_val = 0.010
tau_ref_val = 0.002
J_val = 0.0001
g_val = 4.0
RI_val = 0.030
N_E, N_I, C_E, C_I = 100_000, 25_000, 1_000, 250

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "01_Original_Colab").is_dir()), Path.cwd())
RESULTS = ROOT / "03_Gaussian_Approximation" / "results"
FIGURES = ROOT / "03_Gaussian_Approximation" / "figures"
RESULTS.mkdir(parents=True, exist_ok=True); FIGURES.mkdir(parents=True, exist_ok=True)
np.random.seed(SEED)


## Initial recurrent Poisson input

In [ ]:
def mu_D_stats(nu_in):
    gamma_frac = 0.25
    return RI_val + C_E * J_val * tau_val * (1.0 - gamma_frac * g_val) * nu_in

def initial_poisson_input(nu_in):
    exc_spikes = np.random.poisson(C_E * nu_in * dt_sec, N_steps)
    inh_spikes = np.random.poisson(C_I * nu_in * dt_sec, N_steps)
    return ((exc_spikes * J_val - inh_spikes * g_val * J_val) / dt_sec) * tau_val + RI_val


## Gaussian colored-noise generation

In [ ]:
def generate_colored_noise(S_x_prev, f_prev, nu_in):
    mu = mu_D_stats(nu_in)
    coeff = (C_E * J_val**2 + C_I * (g_val * J_val)**2) * tau_val**2
    f_gen = np.fft.rfftfreq(N_steps, d=dt_sec)
    S_eta = coeff * np.interp(f_gen, f_prev, S_x_prev, left=S_x_prev[0], right=S_x_prev[-1])
    amplitude = np.sqrt(np.maximum(S_eta, 0.0) * N_steps / (2.0 * dt_sec))
    phases = np.random.uniform(0.0, 2.0 * np.pi, len(f_gen))
    X = amplitude * np.exp(1j * phases)
    X[0] = 0.0
    return np.fft.irfft(X, n=N_steps) + mu


## Single-neuron LIF simulation
The readable Python reference loop is the default. A compiled Brian2 alternative is not enabled here because exact spike timing, threshold comparison, and refractory scheduling must first be validated on the same input array; no unverified speedup is substituted.

In [ ]:
def simulate_lif_reference(RI_array):
    v = vR_val
    ref_steps = 0
    spike_indices = []
    dt_over_tau = dt_sec / tau_val
    ref_limit = int(tau_ref_val / dt_sec)
    for n in range(N_steps):
        if ref_steps > 0:
            ref_steps -= 1
            v = vR_val
        else:
            v += (-v + RI_array[n]) * dt_over_tau
            if v >= vT_val:
                spike_indices.append(n)
                v = vR_val
                ref_steps = ref_limit
    return np.asarray(spike_indices, dtype=np.int64) * dt_sec

def psd_from_spikes(spike_times, nperseg=8192):
    x = np.zeros(N_steps)
    indices = (spike_times / dt_sec).astype(int)
    valid = (indices >= 0) & (indices < N_steps)
    np.add.at(x, indices[valid], 1.0 / dt_sec)
    return welch(x, fs=1.0/dt_sec, nperseg=min(N_steps, nperseg), window="hann")


## Iteration loop and diagnostics

In [ ]:
def run_generations():
    np.random.seed(SEED)
    rates, cvs, spectra, frequencies, history = [], [], [], [], []
    previous_spectrum = previous_frequency = None
    started = time.perf_counter()
    for generation in range(1, n_gen + 1):
        if generation == 1:
            RI_input = initial_poisson_input(initial_rate_hz)
        else:
            RI_input = generate_colored_noise(previous_spectrum, previous_frequency, nu_in)
        spikes = simulate_lif_reference(RI_input)
        nu_in = len(spikes) / T_trial
        frequency, spectrum = psd_from_spikes(spikes)
        cv = float(np.std(np.diff(spikes)) / np.mean(np.diff(spikes))) if len(spikes) > 1 else np.nan
        rates.append(nu_in); cvs.append(cv); frequencies.append(frequency); spectra.append(spectrum)
        history.append({"generation": generation, "rate_hz": nu_in, "cv": cv})
        previous_frequency, previous_spectrum = frequency, spectrum
        print(f"Generation {generation}: rate={nu_in:.3f} Hz, CV={cv:.3f}")
    return rates, cvs, frequencies, spectra, history, time.perf_counter() - started

def save_run(rates, cvs, frequencies, spectra, history, runtime_s):
    metadata = {"run_id": RUN_ID, "parameters": {"T_trial_s": T_trial, "dt_s": dt_sec, "generations": n_gen, "initial_rate_hz": initial_rate_hz, "N_E": N_E, "N_I": N_I, "C_E": C_E, "C_I": C_I, "tau_s": tau_val, "threshold_V": vT_val, "reset_V": vR_val, "refractory_s": tau_ref_val, "J_V": J_val, "g": g_val, "RI_V": RI_val}, "seed": SEED, "software": {"python": sys.version, "numpy": np.__version__, "scipy": scipy_version, "platform": platform.platform()}, "analysis": {"estimator": "scipy.signal.welch", "window": "hann", "nperseg": 8192, "spectrum": "one-sided PSD"}, "measurement_duration_s": T_trial, "discarded_transient_s": 0.0, "runtime_s": runtime_s, "history": history}
    np.savez_compressed(RESULTS / f"{RUN_ID}.npz", rates_hz=np.asarray(rates), cvs=np.asarray(cvs), frequencies_hz=np.asarray(frequencies), spectra=np.asarray(spectra))
    (RESULTS / f"{RUN_ID}.json").write_text(json.dumps(metadata, indent=2))

if RUN_SIMULATION:
    rates, cvs, frequencies, spectra, history, runtime_s = run_generations()
    fig, ax = plt.subplots(figsize=(10, 4)); ax.semilogy(frequencies[-1], spectra[-1], color="royalblue")
    ax.set(xlabel="Frequency (Hz)", ylabel="One-sided PSD", title=f"Gaussian approximation, initial rate {initial_rate_hz:g} Hz")
    if SAVE_FIGURES: fig.savefig(FIGURES / f"{RUN_ID}.png", dpi=150)
    plt.show(); save_run(rates, cvs, frequencies, spectra, history, runtime_s)
else:
    print("Simulation disabled. Set RUN_SIMULATION = True for the 500-second iteration.")


## Plots and saved outputs

Each generation is saved, including its firing rate, CV, Welch frequency grid, and spectrum. The `history` metadata is a convergence diagnostic only; it does not trigger early stopping or alter the update rule.